In [31]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nih-chest-xrays/data")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/nih-chest-xrays/data


In [32]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import densenet121, DenseNet121_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

In [33]:
import os

DATA_DIR = "/kaggle/input/datasets/nih-chest-xrays/data"

CSV_PATH = os.path.join(
    DATA_DIR,
    "Data_Entry_2017.csv"
)

IMAGE_DIRS = [
    os.path.join(DATA_DIR, f"images_{i:03d}", "images")
    for i in range(1, 13)
]

print("CSV:", CSV_PATH)

for directory in IMAGE_DIRS:
    print(directory, "->", os.path.exists(directory))

CSV: /kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv
/kaggle/input/datasets/nih-chest-xrays/data/images_001/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_002/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_003/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_004/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_005/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_006/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_007/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_008/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_009/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_010/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_011/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_012/images -> True


In [34]:
image_paths = {}

for image_dir in IMAGE_DIRS:
    for image_name in os.listdir(image_dir):
        image_paths[image_name] = os.path.join(
            image_dir,
            image_name
        )

print("Total images found:", len(image_paths))

Total images found: 112120


In [35]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (112120, 12)


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [36]:
missing_images = [
    image_name
    for image_name in df["Image Index"]
    if image_name not in image_paths
]

print("Missing images:", len(missing_images))

if len(missing_images) > 0:
    print(missing_images[:10])

Missing images: 0


In [37]:
from torch.utils.data import Dataset
from PIL import Image
import torch
import numpy as np


class ChestXrayDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_paths,
        transform=None
    ):

        self.dataframe = dataframe.reset_index(drop=True)
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_name = row["Image Index"]

        image_path = self.image_paths[image_name]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(
            row[diseases].values.astype(np.float32)
        )

        return image, labels

In [38]:
classes = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax",
    "No Finding"
]

print("Number of classes:", len(classes))

Number of classes: 15


In [39]:
for class_name in classes:
    df[class_name] = df["Finding Labels"].apply(
        lambda x: 1 if class_name in x.split("|") else 0
    )

In [40]:
print(df[classes].sum().sort_values())

Hernia                  227
Pneumonia              1431
Fibrosis               1686
Edema                  2303
Emphysema              2516
Cardiomegaly           2776
Pleural_Thickening     3385
Consolidation          4667
Pneumothorax           5302
Mass                   5782
Nodule                 6331
Atelectasis           11559
Effusion              13317
Infiltration          19894
No Finding            60361
dtype: int64


In [41]:
from sklearn.model_selection import train_test_split

patients = df["Patient ID"].unique()

train_patients, temp_patients = train_test_split(
    patients,
    test_size=0.20,
    random_state=42
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=42
)

train_df = df[df["Patient ID"].isin(train_patients)].copy()
val_df = df[df["Patient ID"].isin(val_patients)].copy()
test_df = df[df["Patient ID"].isin(test_patients)].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 89826
Validation: 10930
Test: 11364


In [42]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [43]:
from torch.utils.data import Dataset
from PIL import Image
import torch
import numpy as np

class ChestXrayDataset(Dataset):

    def __init__(self, dataframe, image_paths, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_name = row["Image Index"]
        image_path = self.image_paths[image_name]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(
            row[classes].values.astype(np.float32)
        )

        return image, labels

In [44]:
train_dataset = ChestXrayDataset(
    train_df,
    image_paths,
    train_transform
)

val_dataset = ChestXrayDataset(
    val_df,
    image_paths,
    val_transform
)

test_dataset = ChestXrayDataset(
    test_df,
    image_paths,
    val_transform
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 89826
Validation: 10930
Test: 11364


In [45]:
image, label = train_dataset[0]

print("Image shape:", image.shape)
print("Label shape:", label.shape)
print("Labels:", label)

Image shape: torch.Size([3, 224, 224])
Label shape: torch.Size([15])
Labels: tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


In [46]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("DataLoaders recreated successfully.")
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

DataLoaders recreated successfully.
Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32, 15])


In [47]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32, 15])


In [48]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [49]:
import torch
import torch.nn as nn
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

# Try loading pretrained weights
weights = ConvNeXt_Tiny_Weights.DEFAULT

convnext = convnext_tiny(weights=weights)

num_features = convnext.classifier[2].in_features

convnext.classifier[2] = nn.Linear(
    num_features,
    len(classes)
)

convnext = convnext.to(device)

print("ConvNeXt-Tiny loaded successfully!")
print("Classes:", len(classes))
print("Device:", device)

ConvNeXt-Tiny loaded successfully!
Classes: 15
Device: cuda


In [50]:
import torch

# Calculate class weights from the training set
positive_counts = train_df[classes].sum().values
negative_counts = len(train_df) - positive_counts

pos_weights = negative_counts / positive_counts

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float32
).to(device)

print("pos_weights created successfully!")
print("Shape:", pos_weights.shape)
print("Device:", pos_weights.device)

pos_weights created successfully!
Shape: torch.Size([15])
Device: cuda:0


In [51]:
'''criterion_convnext = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights
)

optimizer_convnext = torch.optim.AdamW(
    convnext.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler_convnext = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_convnext,
    mode="max",
    factor=0.5,
    patience=2
)

from torch.amp import autocast, GradScaler

scaler_convnext = GradScaler("cuda")

print("ConvNeXt training setup ready!")'''

'criterion_convnext = nn.BCEWithLogitsLoss(\n    pos_weight=pos_weights\n)\n\noptimizer_convnext = torch.optim.AdamW(\n    convnext.parameters(),\n    lr=1e-4,\n    weight_decay=1e-4\n)\n\nscheduler_convnext = torch.optim.lr_scheduler.ReduceLROnPlateau(\n    optimizer_convnext,\n    mode="max",\n    factor=0.5,\n    patience=2\n)\n\nfrom torch.amp import autocast, GradScaler\n\nscaler_convnext = GradScaler("cuda")\n\nprint("ConvNeXt training setup ready!")'

In [52]:
'''from tqdm.auto import tqdm
import numpy as np
from sklearn.metrics import roc_auc_score

NUM_EPOCHS_CONVNEXT = 8

best_val_auc_convnext = 0.0

for epoch in range(NUM_EPOCHS_CONVNEXT):

    # =====================================
    # TRAIN
    # =====================================

    convnext.train()

    running_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"ConvNeXt-Tiny Epoch {epoch+1}/{NUM_EPOCHS_CONVNEXT}"
    )

    for images, labels in progress:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_convnext.zero_grad(
            set_to_none=True
        )

        # Mixed precision
        with autocast("cuda"):

            outputs = convnext(images)

            loss = criterion_convnext(
                outputs,
                labels
            )

        # Backpropagation
        scaler_convnext.scale(
            loss
        ).backward()

        # Gradient clipping
        scaler_convnext.unscale_(
            optimizer_convnext
        )

        torch.nn.utils.clip_grad_norm_(
            convnext.parameters(),
            max_norm=1.0
        )

        scaler_convnext.step(
            optimizer_convnext
        )

        scaler_convnext.update()

        running_loss += loss.item()

        # Live loss
        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = (
        running_loss /
        len(train_loader)
    )

    # =====================================
    # VALIDATION
    # =====================================

    convnext.eval()

    val_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with autocast("cuda"):

                outputs = convnext(images)

                loss = criterion_convnext(
                    outputs,
                    labels
                )

            val_loss += loss.item()

            probabilities = torch.sigmoid(
                outputs
            )

            all_preds.append(
                probabilities.cpu().numpy()
            )

            all_labels.append(
                labels.cpu().numpy()
            )

    val_loss /= len(val_loader)

    all_preds = np.concatenate(
        all_preds
    )

    all_labels = np.concatenate(
        all_labels
    )

    # Macro ROC-AUC
    val_auc = roc_auc_score(
        all_labels,
        all_preds,
        average="macro"
    )

    # Scheduler
    scheduler_convnext.step(
        val_auc
    )

    # =====================================
    # RESULTS
    # =====================================

    print(
        f"\n{'='*60}"
    )

    print(
        f"ConvNeXt-Tiny - Epoch "
        f"{epoch+1}/{NUM_EPOCHS_CONVNEXT}"
    )

    print(
        f"Train Loss : {train_loss:.4f}"
    )

    print(
        f"Val Loss   : {val_loss:.4f}"
    )

    print(
        f"Val ROC-AUC: {val_auc:.4f}"
    )

    print(
        f"{'='*60}"
    )

    # =====================================
    # SAVE BEST MODEL
    # =====================================

    if val_auc > best_val_auc_convnext:

        best_val_auc_convnext = val_auc

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": convnext.state_dict(),
                "optimizer_state_dict": optimizer_convnext.state_dict(),
                "val_auc": val_auc,
                "classes": classes
            },
            "/kaggle/working/best_convnext_tiny.pth"
        )

        print(
            "🔥 BEST CONVNEXT-TINY MODEL SAVED!"
        )'''

'from tqdm.auto import tqdm\nimport numpy as np\nfrom sklearn.metrics import roc_auc_score\n\nNUM_EPOCHS_CONVNEXT = 8\n\nbest_val_auc_convnext = 0.0\n\nfor epoch in range(NUM_EPOCHS_CONVNEXT):\n\n    # =====================================\n    # TRAIN\n    # =====================================\n\n    convnext.train()\n\n    running_loss = 0.0\n\n    progress = tqdm(\n        train_loader,\n        desc=f"ConvNeXt-Tiny Epoch {epoch+1}/{NUM_EPOCHS_CONVNEXT}"\n    )\n\n    for images, labels in progress:\n\n        images = images.to(\n            device,\n            non_blocking=True\n        )\n\n        labels = labels.to(\n            device,\n            non_blocking=True\n        )\n\n        optimizer_convnext.zero_grad(\n            set_to_none=True\n        )\n\n        # Mixed precision\n        with autocast("cuda"):\n\n            outputs = convnext(images)\n\n            loss = criterion_convnext(\n                outputs,\n                labels\n            )\n\n       

In [53]:
import torch
import torch.nn as nn
import numpy as np

from torchvision.models import (
    convnext_tiny
)

from sklearn.metrics import roc_auc_score

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [54]:
# ============================================================
# PATHS
# ============================================================

CONVNEXT_PATH = "/kaggle/input/models/yuseifudo12/best-convnext-tiny/pytorch/default/1/best_convnext_tiny.pth"

# ============================================================
# 1. CONVNEXT-TINY
# ============================================================

convnext_ens = convnext_tiny(weights=None)

num_features = convnext_ens.classifier[2].in_features

convnext_ens.classifier[2] = nn.Linear(
    num_features,
    len(classes)
)

checkpoint = torch.load(
    CONVNEXT_PATH,
    map_location=device,
    weights_only=False
)

if "model_state_dict" in checkpoint:
    convnext_ens.load_state_dict(
        checkpoint["model_state_dict"]
    )
else:
    convnext_ens.load_state_dict(checkpoint)

convnext_ens = convnext_ens.to(device)
convnext_ens.eval()

print("✅ ConvNeXt loaded")

✅ ConvNeXt loaded


In [57]:
from tqdm.auto import tqdm

all_labels = []

convnext_preds = []
with torch.no_grad():

    for images, labels in tqdm(
        val_loader,
        desc="Generating ensemble predictions"
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        # -------------------------
        # ConvNeXt
        # -------------------------

        with torch.amp.autocast("cuda"):

            convnext_output = convnext_ens(
                images
            )

        convnext_prob = torch.sigmoid(
            convnext_output
        )
        convnext_preds.append(
            convnext_prob.cpu().numpy()
        )

        all_labels.append(
            labels.numpy()
        )


# Convert to arrays

convnext_preds = np.concatenate(
    convnext_preds
)


all_labels = np.concatenate(
    all_labels
)


# ============================================================
# INDIVIDUAL AUC
# ============================================================

auc_convnext = roc_auc_score(
    all_labels,
    convnext_preds,
    average="macro"
)


# ============================================================
# SIMPLE ENSEMBLE
# ============================================================

ensemble_preds = (
    convnext_preds 
)


auc_ensemble = roc_auc_score(
    all_labels,
    ensemble_preds,
    average="macro"
)


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 60)
print("ENSEMBLE RESULTS")
print("=" * 60)

print(
    f"ConvNeXt-Tiny    : {auc_convnext:.4f}"
)
print("=" * 60)

Generating ensemble predictions:   0%|          | 0/342 [00:00<?, ?it/s]


ENSEMBLE RESULTS
ConvNeXt-Tiny    : 0.8321


In [59]:
import torch
import torch.nn as nn
from torchvision.models import convnext_tiny

# Best checkpoint from Epoch 4
CONVNEXT_BEST = "/kaggle/input/models/yuseifudo12/best-convnext-tiny/pytorch/default/1/best_convnext_tiny.pth"

# Recreate model
convnext_ft = convnext_tiny(weights=None)

num_features = convnext_ft.classifier[2].in_features

convnext_ft.classifier[2] = nn.Linear(
    num_features,
    len(classes)
)

# Load best checkpoint
checkpoint = torch.load(
    CONVNEXT_BEST,
    map_location=device,
    weights_only=False
)

convnext_ft.load_state_dict(
    checkpoint["model_state_dict"]
)

convnext_ft = convnext_ft.to(device)

print("✅ Best ConvNeXt-Tiny checkpoint loaded")
print("Starting Val AUC:", checkpoint["val_auc"])

✅ Best ConvNeXt-Tiny checkpoint loaded
Starting Val AUC: 0.8321084277773712


In [60]:
from torch.amp import autocast, GradScaler

criterion_convnext_ft = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights
)

optimizer_convnext_ft = torch.optim.AdamW(
    convnext_ft.parameters(),
    lr=3e-6,
    weight_decay=1e-4
)

scheduler_convnext_ft = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_convnext_ft,
    mode="max",
    factor=0.5,
    patience=1
)

scaler_convnext_ft = GradScaler("cuda")

print("🔥 ConvNeXt fine-tuning setup ready!")

🔥 ConvNeXt fine-tuning setup ready!


In [61]:
from tqdm.auto import tqdm
import numpy as np
from sklearn.metrics import roc_auc_score

NUM_EPOCHS_FT = 3

best_val_auc_ft = 0.8321

for epoch in range(NUM_EPOCHS_FT):

    # =========================
    # TRAIN
    # =========================

    convnext_ft.train()

    running_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"ConvNeXt Fine-Tune {epoch+1}/{NUM_EPOCHS_FT}"
    )

    for images, labels in progress:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_convnext_ft.zero_grad(
            set_to_none=True
        )

        with autocast("cuda"):

            outputs = convnext_ft(images)

            loss = criterion_convnext_ft(
                outputs,
                labels
            )

        scaler_convnext_ft.scale(
            loss
        ).backward()

        scaler_convnext_ft.unscale_(
            optimizer_convnext_ft
        )

        torch.nn.utils.clip_grad_norm_(
            convnext_ft.parameters(),
            max_norm=1.0
        )

        scaler_convnext_ft.step(
            optimizer_convnext_ft
        )

        scaler_convnext_ft.update()

        running_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = (
        running_loss /
        len(train_loader)
    )

    # =========================
    # VALIDATION
    # =========================

    convnext_ft.eval()

    val_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with autocast("cuda"):

                outputs = convnext_ft(images)

                loss = criterion_convnext_ft(
                    outputs,
                    labels
                )

            val_loss += loss.item()

            probs = torch.sigmoid(outputs)

            all_preds.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                labels.cpu().numpy()
            )

    val_loss /= len(val_loader)

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    val_auc = roc_auc_score(
        all_labels,
        all_preds,
        average="macro"
    )

    scheduler_convnext_ft.step(val_auc)

    # =========================
    # RESULTS
    # =========================

    print("\n" + "=" * 60)

    print(
        f"ConvNeXt Fine-Tuning - "
        f"Epoch {epoch+1}/{NUM_EPOCHS_FT}"
    )

    print(
        f"Train Loss : {train_loss:.4f}"
    )

    print(
        f"Val Loss   : {val_loss:.4f}"
    )

    print(
        f"Val ROC-AUC: {val_auc:.4f}"
    )

    print("=" * 60)

    # =========================
    # SAVE BEST
    # =========================

    if val_auc > best_val_auc_ft:

        best_val_auc_ft = val_auc

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": convnext_ft.state_dict(),
                "optimizer_state_dict": optimizer_convnext_ft.state_dict(),
                "val_auc": val_auc,
                "classes": classes
            },
            "/kaggle/working/best_convnext_tiny_finetuned.pth"
        )

        print(
            "🔥 NEW BEST CONVNEXT-TINY "
            "FINE-TUNED MODEL SAVED!"
        )

ConvNeXt Fine-Tune 1/3:   0%|          | 0/2808 [00:00<?, ?it/s]


ConvNeXt Fine-Tuning - Epoch 1/3
Train Loss : 0.7465
Val Loss   : 0.9962
Val ROC-AUC: 0.8356
🔥 NEW BEST CONVNEXT-TINY FINE-TUNED MODEL SAVED!


ConvNeXt Fine-Tune 2/3:   0%|          | 0/2808 [00:00<?, ?it/s]


ConvNeXt Fine-Tuning - Epoch 2/3
Train Loss : 0.7210
Val Loss   : 1.0047
Val ROC-AUC: 0.8359
🔥 NEW BEST CONVNEXT-TINY FINE-TUNED MODEL SAVED!


ConvNeXt Fine-Tune 3/3:   0%|          | 0/2808 [00:00<?, ?it/s]


ConvNeXt Fine-Tuning - Epoch 3/3
Train Loss : 0.7104
Val Loss   : 1.0233
Val ROC-AUC: 0.8358


In [62]:
import torch
import torch.nn as nn
import numpy as np

from torchvision.models import convnext_tiny
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score
)

# ==========================================
# BEST FINE-TUNED CHECKPOINT
# ==========================================

BEST_CONVNEXT_PATH = "/kaggle/working/best_convnext_tiny_finetuned.pth"

checkpoint = torch.load(
    BEST_CONVNEXT_PATH,
    map_location=device,
    weights_only=False
)

print("Checkpoint Val AUC:", checkpoint.get("val_auc", "N/A"))
print("Checkpoint Epoch:", checkpoint.get("epoch", "N/A"))

# ==========================================
# RECREATE MODEL
# ==========================================

convnext_final = convnext_tiny(weights=None)

num_features = convnext_final.classifier[2].in_features

convnext_final.classifier[2] = nn.Linear(
    num_features,
    len(classes)
)

convnext_final.load_state_dict(
    checkpoint["model_state_dict"]
)

convnext_final = convnext_final.to(device)
convnext_final.eval()

print("\n✅ Best ConvNeXt-Tiny loaded!")
print("Classes:", len(classes))
print("Device:", device)

Checkpoint Val AUC: 0.8359305247769845
Checkpoint Epoch: 2

✅ Best ConvNeXt-Tiny loaded!
Classes: 15
Device: cuda


In [63]:
from tqdm.auto import tqdm

val_preds = []
val_labels = []

convnext_final.eval()

with torch.no_grad():

    for images, labels in tqdm(
        val_loader,
        desc="Generating validation predictions"
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast("cuda"):

            outputs = convnext_final(images)

        probabilities = torch.sigmoid(outputs)

        val_preds.append(
            probabilities.cpu().numpy()
        )

        val_labels.append(
            labels.numpy()
        )

val_preds = np.concatenate(val_preds)
val_labels = np.concatenate(val_labels)

print("\n✅ Validation predictions generated!")

print("Predictions shape:", val_preds.shape)
print("Labels shape:", val_labels.shape)

Generating validation predictions:   0%|          | 0/342 [00:00<?, ?it/s]


✅ Validation predictions generated!
Predictions shape: (10930, 15)
Labels shape: (10930, 15)


In [66]:
val_auc = roc_auc_score(
    val_labels,
    val_preds,
    average="macro"
)

print(
    f"🔥 Validation Macro ROC-AUC: {val_auc:.4f}"
)

🔥 Validation Macro ROC-AUC: 0.8359


In [67]:
thresholds = np.arange(
    0.05,
    0.96,
    0.05
)

best_thresholds = []
best_f1_scores = []

for i, disease in enumerate(classes):

    best_threshold = 0.5
    best_f1 = 0.0

    for threshold in thresholds:

        predictions = (
            val_preds[:, i] >= threshold
        ).astype(int)

        f1 = f1_score(
            val_labels[:, i],
            predictions,
            zero_division=0
        )

        if f1 > best_f1:

            best_f1 = f1
            best_threshold = threshold

    best_thresholds.append(best_threshold)
    best_f1_scores.append(best_f1)

# Convert to arrays
best_thresholds = np.array(best_thresholds)
best_f1_scores = np.array(best_f1_scores)

print("\n" + "=" * 65)
print("OPTIMIZED DISEASE THRESHOLDS")
print("=" * 65)

for disease, threshold, f1 in zip(
    classes,
    best_thresholds,
    best_f1_scores
):

    print(
        f"{disease:22s} "
        f"Threshold: {threshold:.2f} "
        f"F1: {f1:.4f}"
    )

print("=" * 65)


OPTIMIZED DISEASE THRESHOLDS
Atelectasis            Threshold: 0.70 F1: 0.3942
Cardiomegaly           Threshold: 0.90 F1: 0.4604
Consolidation          Threshold: 0.85 F1: 0.2363
Edema                  Threshold: 0.90 F1: 0.2436
Effusion               Threshold: 0.80 F1: 0.5201
Emphysema              Threshold: 0.85 F1: 0.4800
Fibrosis               Threshold: 0.85 F1: 0.1826
Hernia                 Threshold: 0.95 F1: 0.4000
Infiltration           Threshold: 0.55 F1: 0.4113
Mass                   Threshold: 0.85 F1: 0.3864
Nodule                 Threshold: 0.85 F1: 0.2974
Pleural_Thickening     Threshold: 0.80 F1: 0.2063
Pneumonia              Threshold: 0.80 F1: 0.0996
Pneumothorax           Threshold: 0.90 F1: 0.3982
No Finding             Threshold: 0.30 F1: 0.7588


In [68]:
# Default 0.5 thresholds
default_predictions = (
    val_preds >= 0.5
).astype(int)

# Optimized thresholds
optimized_predictions = (
    val_preds >= best_thresholds
).astype(int)

default_f1 = f1_score(
    val_labels,
    default_predictions,
    average="macro",
    zero_division=0
)

optimized_f1 = f1_score(
    val_labels,
    optimized_predictions,
    average="macro",
    zero_division=0
)

print(
    f"Default F1 (0.5):     {default_f1:.4f}"
)

print(
    f"Optimized F1:         {optimized_f1:.4f}"
)

print(
    f"Improvement:          "
    f"{optimized_f1 - default_f1:+.4f}"
)

Default F1 (0.5):     0.2965
Optimized F1:         0.3650
Improvement:          +0.0685


In [69]:
print("\n" + "=" * 75)
print("PER-CLASS PERFORMANCE")
print("=" * 75)

for i, disease in enumerate(classes):

    auc = roc_auc_score(
        val_labels[:, i],
        val_preds[:, i]
    )

    preds = optimized_predictions[:, i]

    f1 = f1_score(
        val_labels[:, i],
        preds,
        zero_division=0
    )

    precision = precision_score(
        val_labels[:, i],
        preds,
        zero_division=0
    )

    recall = recall_score(
        val_labels[:, i],
        preds,
        zero_division=0
    )

    print(
        f"{disease:22s} "
        f"AUC: {auc:.4f} | "
        f"F1: {f1:.4f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f}"
    )

print("=" * 75)


PER-CLASS PERFORMANCE
Atelectasis            AUC: 0.8172 | F1: 0.3942 | Precision: 0.3107 | Recall: 0.5390
Cardiomegaly           AUC: 0.9312 | F1: 0.4604 | Precision: 0.3665 | Recall: 0.6190
Consolidation          AUC: 0.8250 | F1: 0.2363 | Precision: 0.2137 | Recall: 0.2643
Edema                  AUC: 0.8935 | F1: 0.2436 | Precision: 0.1818 | Recall: 0.3689
Effusion               AUC: 0.8804 | F1: 0.5201 | Precision: 0.4867 | Recall: 0.5583
Emphysema              AUC: 0.9314 | F1: 0.4800 | Precision: 0.4021 | Recall: 0.5952
Fibrosis               AUC: 0.8135 | F1: 0.1826 | Precision: 0.1486 | Recall: 0.2366
Hernia                 AUC: 0.8925 | F1: 0.4000 | Precision: 0.4286 | Recall: 0.3750
Infiltration           AUC: 0.7120 | F1: 0.4113 | Precision: 0.3496 | Recall: 0.4995
Mass                   AUC: 0.8484 | F1: 0.3864 | Precision: 0.3721 | Recall: 0.4019
Nodule                 AUC: 0.7704 | F1: 0.2974 | Precision: 0.3235 | Recall: 0.2752
Pleural_Thickening     AUC: 0.8057 | F1: 0

In [70]:
test_preds = []
test_labels = []

convnext_final.eval()

with torch.no_grad():

    for images, labels in tqdm(
        test_loader,
        desc="Generating TEST predictions"
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast("cuda"):

            outputs = convnext_final(images)

        probabilities = torch.sigmoid(
            outputs
        )

        test_preds.append(
            probabilities.cpu().numpy()
        )

        test_labels.append(
            labels.numpy()
        )

test_preds = np.concatenate(test_preds)
test_labels = np.concatenate(test_labels)

print("\n🔥 TEST predictions generated!")
print("Shape:", test_preds.shape)

Generating TEST predictions:   0%|          | 0/356 [00:00<?, ?it/s]


🔥 TEST predictions generated!
Shape: (11364, 15)


In [71]:
test_auc = roc_auc_score(
    test_labels,
    test_preds,
    average="macro"
)

print("\n" + "=" * 60)
print("FINAL TEST RESULT")
print("=" * 60)

print(
    f"🔥 TEST MACRO ROC-AUC: {test_auc:.4f}"
)

print("=" * 60)


FINAL TEST RESULT
🔥 TEST MACRO ROC-AUC: 0.8400
